# ProGen3 Protein Sequence Generation and Scoring

![ProGen3 Protein Sequence Generation and Scoring](https://proto-bio.github.io/proto-assets/images/tool/progen3/hero.png)

This notebook demonstrates both functions of the ProGen3 tool. In the first section, we use `run_progen3_sample` to generate protein sequences using forward, reverse, and unconditional generation modes. In the second section, we use `run_progen3_score` to compute bidirectional log-likelihood and perplexity metrics that measure how natural a sequence appears to the model.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("progen3")
display_overview("progen3")
display_docs_section("progen3", "Background")

## Available tools

In [ ]:
display_available_tools("progen3")

### `run_progen3_sample`

ProGen3 is a Mixture-of-Experts protein language model that supports both forward (N-to-C) and reverse (C-to-N) autoregressive generation. Forward mode extends a sequence from an N-terminal prompt, while reverse mode extends from a C-terminal prompt — useful when a conserved C-terminal motif or membrane anchor must be preserved. Passing an empty string as the prompt produces unconditional generation, where the model samples an entire sequence from scratch. Available checkpoints range from 112M to 3B parameters.

In [ ]:
from proto_tools import (
    ProGen3SampleInput,
    ProGen3SampleConfig,
    run_progen3_sample,
)

In [ ]:
# Display docs
display_api_reference("progen3", "input", "run_progen3_sample")

# Forward generation from an N-terminal prompt
inputs = ProGen3SampleInput(prompts=["MKTLVIVTGA"])

In [ ]:
# Display config docs
display_api_reference("progen3", "config", "run_progen3_sample")

# Configure forward sampling | see docs above for all fields
config = ProGen3SampleConfig(
    model_checkpoint="progen3-762m",
    direction="forward",
    temperature=0.2,
    top_p=0.95,
    max_new_tokens=100,
    device="cuda",  # ProGen3 requires a GPU; device="cpu" is rejected
)

In [ ]:
# Run the sampling function
result = run_progen3_sample(inputs, config)

In [ ]:
# Display output docs
display_api_reference("progen3", "output", "run_progen3_sample")

# Show the generated sequence
print(f"Generated {len(result.sequences)} sequence(s)")
print(f"Prompt:   {inputs.prompts[0]}")
print(f"Sequence: {result.sequences[0][:80]}...")

### Reverse Generation

Setting `direction="reverse"` causes the model to extend a C-terminal prompt toward the N-terminus. This is useful when the design constraint is at the end of the protein, such as a purification tag, a membrane anchor, or a conserved C-terminal domain boundary.

In [ ]:
# Generate from a C-terminal prompt in reverse direction
inputs_rev = ProGen3SampleInput(prompts=["RYTEAFLK"])
config_rev = ProGen3SampleConfig(
    model_checkpoint="progen3-762m",
    direction="reverse",
    max_new_tokens=30,
    device="cuda",  # ProGen3 requires a GPU; device="cpu" is rejected
)

In [ ]:
result_rev = run_progen3_sample(inputs_rev, config_rev)
print(f"Prompt:" + " "*len(result_rev.sequences[0]) + "<-" + inputs_rev.prompts[0])
print(f"Generated:       {result_rev.sequences[0]}")

### Unconditional Generation

Passing an empty string as the prompt produces unconditional generation, where the model samples an entire protein sequence from scratch without any seed context. This is useful for exploring the space of plausible proteins or for generating diverse starting points that can be refined with downstream tools.

In [ ]:
# Unconditional generation with an empty prompt
inputs_uncond = ProGen3SampleInput(prompts=[""])
config_uncond = ProGen3SampleConfig(
    max_new_tokens=200,
    temperature=0.8,
    device="cuda",  # ProGen3 requires a GPU; device="cpu" is rejected
)

result_uncond = run_progen3_sample(inputs_uncond, config_uncond)
print(f"Generated: {result_uncond.sequences[0][:80]}...")
print(f"Length: {len(result_uncond.sequences[0])} residues")

### `run_progen3_score`

ProGen3 scoring computes bidirectional log-likelihoods by averaging the forward (N-to-C) and reverse (C-to-N) autoregressive probabilities at each position. This bidirectional approach produces a more robust naturalness estimate than unidirectional models. The resulting perplexity measures how consistent a sequence is with the patterns learned from natural proteins, making it useful for ranking design candidates, comparing wild-type and mutant variants, or filtering sequences before expensive experimental validation.

In [ ]:
from proto_tools import (
    ProGen3ScoringInput,
    ProGen3ScoringConfig,
    run_progen3_score,
)

In [ ]:
# Display docs
display_api_reference("progen3", "input", "run_progen3_score")

# Score a hemoglobin alpha fragment and a signal peptide
score_inputs = ProGen3ScoringInput(
    sequences=[
        "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHF",  # Hemoglobin alpha fragment
        "MKTLLILAVVAAALA",  # Signal peptide
    ]
)

In [ ]:
# Display config docs
display_api_reference("progen3", "config", "run_progen3_score")

# Configure scoring | see docs above for all fields
score_config = ProGen3ScoringConfig(
    model_checkpoint="progen3-762m",
    device="cuda",  # ProGen3 requires a GPU; device="cpu" is rejected
)

In [ ]:
# Run the scoring function
score_result = run_progen3_score(score_inputs, score_config)

In [ ]:
# Display output docs
display_api_reference("progen3", "output", "run_progen3_score")

# Show scoring results for each input sequence
for i, score in enumerate(score_result.scores):
    print(f"Sequence {i + 1}: {score_inputs.sequences[i][:40]}...")
    print(f"    Log-likelihood:     {score['log_likelihood']:.4f}")
    print(f"    Avg log-likelihood: {score['avg_log_likelihood']:.4f}")
    print(f"    Perplexity:         {score['perplexity']:.4f}")

In [ ]:
for i, score in enumerate(score_result.scores):
    # Per-position metrics: forward, reverse, and bidirectional log-likelihoods.
    # These live on the Metrics container as ``_pp``-suffixed extras.
    if "log_likelihood_pp" in score:
        seq = score_inputs.sequences[i]
        print(f"\n    Per-position log-likelihoods (first 10 residues):")
        print(f"    {'Pos':>4} {'AA':>3} {'Forward':>9} {'Reverse':>9} {'Bidir':>9}")
        for j in range(min(10, len(seq))):
            fwd = score["forward_log_likelihood_pp"][j]
            rev = score["reverse_log_likelihood_pp"][j]
            bid = score["log_likelihood_pp"][j]
            fwd_str = f"{fwd:.4f}" if fwd is not None else "N/A"
            rev_str = f"{rev:.4f}" if rev is not None else "N/A"
            bid_str = f"{bid:.4f}" if bid is not None else "N/A"
            print(f"    {j + 1:>4} {seq[j]:>3} {fwd_str:>9} {rev_str:>9} {bid_str:>9}")
    print()

## Export Results

Generated sequences and scoring metrics can be saved to standard file formats for downstream use. Sampling results support FASTA, TXT, and JSON export. Scoring results support CSV and JSON export.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="progen3_sequences", export_path=output_dir, file_format="fasta")
print(f"Generated sequences exported to {output_dir / 'progen3_sequences.fasta'}")

score_result.export(name="progen3_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'progen3_scores.csv'}")